In [67]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

In [68]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(df.info())
print(df.head())
print(df.tail())
print(df.describe())

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [69]:
df = df.drop('customerID', axis=1)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges']=df['TotalCharges'].fillna(0)
print('TotalCharges missing after conversion:', df['TotalCharges'].isna().sum())
print(df.dtypes)

TotalCharges missing after conversion: 0
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
dtype: object


In [70]:
x=df.drop('Churn',axis=1)
y=df['Churn']
y=y.map({"No":0,"Yes":1})
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,shuffle=True)
numeric_cols = x_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = x_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric columns:")
print(numeric_cols)

print("\nCategorical columns:")
print(categorical_cols)


Numeric columns:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical columns:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


/tmp/ipykernel_2317/4149193170.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = x_train.select_dtypes(include=["object", "category"]).columns.tolist()


In [71]:
preprocessor=ColumnTransformer(transformers=[("num",StandardScaler(),numeric_cols),("cat",OneHotEncoder(handle_unknown="ignore"),categorical_cols)
]
)
print(x_train["TotalCharges"].dtype)
x_train_processed=preprocessor.fit_transform(x_train)
x_test_processed=preprocessor.transform(x_test)


float64


In [72]:
pipeline=Pipeline(steps=[("preprocessing",preprocessor),("model",LogisticRegression(max_iter=1000,class_weight="balanced"))])
print(pipeline)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['SeniorCitizen', 'tenure',
                                                   'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['gender', 'Partner',
                                                   'Dependents', 'PhoneService',
                                                   'MultipleLines',
                                                   'InternetService',
                                                   'OnlineSecurity',
                                                   'OnlineBackup',
                                                   'DeviceProtection',
                              

In [73]:
pipeline.fit(x_train,y_train)
pipeline.score(x_test,y_test)
y_pred=pipeline.predict(x_test)
print(confusion_matrix(y_test,y_pred))
print()
print(classification_report(y_test,y_pred))
y_probs=pipeline.predict_proba(x_test)[:,1]
x_train_final, x_val, y_train_final, y_val = train_test_split(
    x_train,
    y_train,
    test_size=0.25,
    random_state=42,
    stratify=y_train
)

[[748 288]
 [ 65 308]]

              precision    recall  f1-score   support

           0       0.92      0.72      0.81      1036
           1       0.52      0.83      0.64       373

    accuracy                           0.75      1409
   macro avg       0.72      0.77      0.72      1409
weighted avg       0.81      0.75      0.76      1409



In [74]:
pipeline.fit(x_train_final, y_train_final)
probs_val=pipeline.predict_proba(x_val)[0:,1]
thresholds = np.linspace(0.01, 0.99, 99)

best_profit = -1e18
best_threshold = None

for t in thresholds:
    y_val_pred = (probs_val >= t).astype(int)
    
    cm = confusion_matrix(y_val, y_val_pred)
    
    if cm.shape != (2,2):
        continue
        
    tn, fp, fn, tp = cm.ravel()
    
    profit = (10000 * tp) - (500 * fp) - (10000 * fn)
    
    if profit > best_profit:
        best_profit = profit
        best_threshold = t

print("Best threshold:", best_threshold)
print("Best validation profit:", best_profit)



Best threshold: 0.12
Best validation profit: 3338000
